# Validação cruzada

**Objetivo:** comparar uma única divisão treino/teste com a validação cruzada k-fold e ver por que a média do CV é mais confiável.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

In [ ]:
from sklearn.datasets import load_iris
X, y = load_iris(return_X_y=True)

## Uma divisão só varia bastante

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

for s in range(5):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=s)
    acc = KNeighborsClassifier().fit(Xtr, ytr).score(Xte, yte)
    print("seed =", s, "-> acuracia =", round(acc, 3))

## k-fold: estimativa estável (com pipeline, sem vazamento)

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

modelo = make_pipeline(StandardScaler(), KNeighborsClassifier())
scores = cross_val_score(modelo, X, y, cv=5)
print("folds:", np.round(scores, 3))
print("CV:", round(scores.mean(), 3), "+/-", round(scores.std(), 3))

## Exercícios

**1.** Rode com cv=10. A média muda muito? E o desvio?

**2.** Use `StratifiedKFold` e confirme que cada fold preserva a proporção das classes.

In [ ]:
# @title Solução
s10 = cross_val_score(modelo, X, y, cv=10)
print("cv=10:", round(s10.mean(), 3), "+/-", round(s10.std(), 3))
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5)
for i, (_, te) in enumerate(skf.split(X, y)):
    vals, cnt = np.unique(y[te], return_counts=True)
    print("fold", i, ":", dict(zip(vals.tolist(), cnt.tolist())))